In [2]:
import pandas as pd
import subprocess
import os

# Load station list
stations = pd.read_csv('isd-history.csv')
oregon = stations[stations['STATE'] == 'OR'].copy()

# Format the filename the way the bucket uses it
oregon['USAF'] = oregon['USAF'].astype(str).str.zfill(6)
oregon['WBAN'] = oregon['WBAN'].astype(str).str.zfill(5)
oregon['filename'] = oregon['USAF'] + oregon['WBAN'] + '.csv'

print(f"Found {len(oregon)} Oregon stations")
print(oregon[['USAF', 'WBAN', 'STATION NAME', 'filename']].to_string())

Found 113 Oregon stations
         USAF   WBAN                                 STATION NAME         filename
14564  690330  99999                            HOOD CANAL BRIDGE  69033099999.csv
17165  720202  00118                            TILLAMOOK AIRPORT  72020200118.csv
17166  720202  99999                                TILLAMOOK AWS  72020299999.csv
17342  720365  24267                            BROOKINGS AIRPORT  72036524267.csv
17343  720365  99999                            BROOKINGS AIRPORT  72036599999.csv
17593  720638  00224                       BEND MUNICIPAL AIRPORT  72063800224.csv
17691  720753  99999                                KEN JERNSTEDT  72075399999.csv
17709  720837  99999                                 JOSEPH STATE  72083799999.csv
20406  725891  99999                                     LAKEVIEW  72589199999.csv
20407  725895  94236                        KLAMATH FALLS AIRPORT  72589594236.csv
20408  725895  99999                                KLAMATH F

In [ ]:
import pandas as pd
import subprocess
import os

# Load station list and filter Oregon
stations = pd.read_csv('isd-history.csv')
oregon = stations[stations['STATE'] == 'OR'].copy()

# Format filenames
oregon['USAF'] = oregon['USAF'].astype(str).str.zfill(6)
oregon['WBAN'] = oregon['WBAN'].astype(str).str.zfill(5)
oregon['filename'] = oregon['USAF'] + oregon['WBAN'] + '.csv'

# Create output folder
os.makedirs('oregon_weather', exist_ok=True)

years = range(1970, 2022)
total = len(years) * len(oregon)
count = 0
skipped = 0

for year in years:
    for _, row in oregon.iterrows():
        filename = row['filename']
        s3_path = f"s3://noaa-gsod-pds/{year}/{filename}"
        local_path = f"oregon_weather/{year}_{filename}"

        # Skip if already downloaded
        if os.path.exists(local_path):
            count += 1
            continue

        result = subprocess.run(
            ["aws", "s3", "cp", "--no-sign-request", s3_path, local_path],
            capture_output=True
        )

        if result.returncode != 0:
            skipped += 1  # Station didn't exist that year, that's normal
        
        count += 1

    print(f" {year} done - {count}/{total} processed, {skipped} missing (normal)")

print(" Download complete! All files saved to oregon_weather/")

✅ 1970 done — 113/5876 processed, 101 missing (normal)
✅ 1971 done — 226/5876 processed, 202 missing (normal)
✅ 1972 done — 339/5876 processed, 304 missing (normal)
✅ 1973 done — 452/5876 processed, 397 missing (normal)
✅ 1974 done — 565/5876 processed, 492 missing (normal)
✅ 1975 done — 678/5876 processed, 582 missing (normal)
✅ 1976 done — 791/5876 processed, 669 missing (normal)
✅ 1977 done — 904/5876 processed, 754 missing (normal)
✅ 1978 done — 1017/5876 processed, 842 missing (normal)
✅ 1979 done — 1130/5876 processed, 930 missing (normal)
✅ 1980 done — 1243/5876 processed, 1016 missing (normal)
✅ 1981 done — 1356/5876 processed, 1105 missing (normal)
✅ 1982 done — 1469/5876 processed, 1193 missing (normal)
✅ 1983 done — 1582/5876 processed, 1281 missing (normal)
✅ 1984 done — 1695/5876 processed, 1367 missing (normal)
✅ 1985 done — 1808/5876 processed, 1454 missing (normal)
✅ 1986 done — 1921/5876 processed, 1542 missing (normal)
✅ 1987 done — 2034/5876 processed, 1629 missing (

In [ ]:
import pandas as pd
import glob
import os

# Find all downloaded files
all_files = glob.glob('oregon_weather/*.csv')
print(f"Found {len(all_files)} files to combine...")

dfs = []
skipped = 0

for i, f in enumerate(all_files):
    try:
        df = pd.read_csv(f)
        
        # Extract year and station from filename
        basename = os.path.basename(f)  # e.g. "1970_72020200118.csv"
        parts = basename.replace('.csv', '').split('_')
        df['YEAR_FILE'] = parts[0]
        df['STATION_FILE'] = parts[1]
        
        dfs.append(df)
    except Exception as e:
        skipped += 1

    if (i + 1) % 500 == 0:
        print(f"  Processed {i+1}/{len(all_files)} files...")

print(f"Combining {len(dfs)} files ({skipped} skipped due to errors)...")
combined = pd.concat(dfs, ignore_index=True)

# Convert DATE to datetime
combined['DATE'] = pd.to_datetime(combined['DATE'], errors='coerce')

# Sort by station and date
combined = combined.sort_values(['STATION_FILE', 'DATE']).reset_index(drop=True)

print(f"   Combined shape: {combined.shape}")
print(f"   Date range: {combined['DATE'].min()} to {combined['DATE'].max()}")
print(f"   Columns: {list(combined.columns)}")

# Save to CSV
output_path = 'oregon_weather_1970_2021.csv'
combined.to_csv(output_path, index=False)
print(f"🎉 Saved to {output_path}!")

Found 1533 files to combine...
  Processed 500/1533 files...
  Processed 1000/1533 files...
  Processed 1500/1533 files...
Combining 1533 files (0 skipped due to errors)...
✅ Combined shape: (509866, 30)
   Date range: 1970-01-01 00:00:00 to 2021-12-31 00:00:00
   Columns: ['STATION', 'DATE', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'TEMP', 'TEMP_ATTRIBUTES', 'DEWP', 'DEWP_ATTRIBUTES', 'SLP', 'SLP_ATTRIBUTES', 'STP', 'STP_ATTRIBUTES', 'VISIB', 'VISIB_ATTRIBUTES', 'WDSP', 'WDSP_ATTRIBUTES', 'MXSPD', 'GUST', 'MAX', 'MAX_ATTRIBUTES', 'MIN', 'MIN_ATTRIBUTES', 'PRCP', 'PRCP_ATTRIBUTES', 'SNDP', 'FRSHTT', 'YEAR_FILE', 'STATION_FILE']
🎉 Saved to oregon_weather_1970_2021.csv!


In [ ]:
import pandas as pd

df = pd.read_csv('oregon_weather_1970_2021.csv', parse_dates=['DATE'])

# Replace NOAA missing flags
missing_values = {
    'TEMP':  9999.9, 'DEWP':  9999.9, 'SLP':   9999.9,
    'STP':   9999.9, 'VISIB': 999.9,  'WDSP':  999.9,
    'MXSPD': 999.9,  'GUST':  999.9,  'MAX':   9999.9,
    'MIN':   9999.9, 'PRCP':  99.99,  'SNDP':  999.9,
}
for col, flag in missing_values.items():
    df[col] = df[col].replace(flag, pd.NA)

# Force numeric types
for col in ['TEMP', 'DEWP', 'MAX', 'MIN', 'PRCP', 'WDSP']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Time columns
df['YEAR']  = df['DATE'].dt.year
df['MONTH'] = df['DATE'].dt.month
df['DOY']   = df['DATE'].dt.dayofyear
df['FIRE_SEASON'] = df['MONTH'].between(6, 10).astype(int)

# Derived features
df['TEMP_RANGE'] = df['MAX'] - df['MIN']
df['VPD_PROXY']  = df['TEMP'] - df['DEWP']

# Rolling 30-day precipitation per station
df = df.sort_values(['STATION', 'DATE'])
df['PRCP_30DAY'] = (
    df.groupby('STATION')['PRCP']
    .transform(lambda x: x.rolling(30, min_periods=1).sum())
)

print(f"Shape: {df.shape}")
print(df[['TEMP','DEWP','WDSP','PRCP','MAX','MIN','TEMP_RANGE','VPD_PROXY','PRCP_30DAY']].describe())

df.to_csv('oregon_weather_cleaned.csv', index=False)
print(" Saved to oregon_weather_cleaned.csv")

Shape: (509866, 37)
                TEMP           DEWP           WDSP           PRCP  \
count  509866.000000  454815.000000  486596.000000  476763.000000   
mean       51.046797      39.301715       5.869436       0.064707   
std        13.172423      10.930963       3.496404       0.231045   
min       -23.300000     -89.600000       0.000000       0.000000   
25%        42.400000      32.300000       3.500000       0.000000   
50%        51.100000      40.300000       5.300000       0.000000   
75%        60.000000      47.500000       7.500000       0.010000   
max       100.100000      74.000000      49.900000      13.390000   

                 MAX            MIN     TEMP_RANGE      VPD_PROXY  \
count  509417.000000  509474.000000  509194.000000  454815.000000   
mean       62.712131      40.852619      21.850242      11.694954   
std        16.164429      11.937578      11.528712       8.964668   
min        -8.900000     -40.900000       0.200000       0.000000   
25%        51

In [4]:
import pandas as pd
import glob

# Load a sample of raw files to check original PRCP values
raw_files = glob.glob('oregon_weather/*.csv')
print(f"Total raw files: {len(raw_files)}")

# Load a sample of 100 files
sample_dfs = []
for f in raw_files[:100]:
    try:
        df = pd.read_csv(f)
        sample_dfs.append(df)
    except:
        pass

raw = pd.concat(sample_dfs, ignore_index=True)
print(f"Sample records: {len(raw):,}")
print(f"\nRaw PRCP column exists: {'PRCP' in raw.columns}")
print(f"\nRaw PRCP value counts (top 10):")
print(raw['PRCP'].value_counts().head(10))

print(f"\nRaw PRCP stats:")
print(raw['PRCP'].describe())

# Check how many are the missing flag (99.99)
missing_flag = (raw['PRCP'] == 99.99).sum()
total = len(raw)
print(f"\nMissing flag (99.99): {missing_flag:,} ({missing_flag/total*100:.1f}%)")
print(f"Zero values:          {(raw['PRCP'] == 0).sum():,} ({(raw['PRCP']==0).mean()*100:.1f}%)")
print(f"Valid non-zero:       {((raw['PRCP'] > 0) & (raw['PRCP'] < 99.99)).sum():,}")

Total raw files: 1533
Sample records: 34,598

Raw PRCP column exists: True

Raw PRCP value counts (top 10):
PRCP
0.00     22850
99.99     3040
0.01      1182
0.02       722
0.04       599
0.03       405
0.08       399
0.05       317
0.12       272
0.06       264
Name: count, dtype: int64

Raw PRCP stats:
count    34598.000000
mean         8.842545
std         28.290713
min          0.000000
25%          0.000000
50%          0.000000
75%          0.050000
max         99.990000
Name: PRCP, dtype: float64

Missing flag (99.99): 3,040 (8.8%)
Zero values:          22,850 (66.0%)
Valid non-zero:       8,708


In [5]:
weather = pd.read_csv('oregon_weather_cleaned.csv', parse_dates=['DATE'])

# Check per station how many PRCP values are NaN vs valid
station_prcp = weather.groupby('STATION')['PRCP'].agg(
    total='count',
    missing=lambda x: x.isna().sum(),
    valid=lambda x: x.notna().sum(),
    all_missing=lambda x: x.isna().all(),
    all_zero=lambda x: (x == 0).all(),
    mean='mean'
)

print(f"Total stations: {station_prcp.shape[0]}")
print(f"\nStations with ALL missing PRCP:  {station_prcp['all_missing'].sum()}")
print(f"Stations with ALL zero PRCP:     {station_prcp['all_zero'].sum()}")
print(f"Stations with any missing PRCP:  {(station_prcp['missing'] > 0).sum()}")

print(f"\nStations where >50% PRCP is missing:")
mostly_missing = station_prcp[station_prcp['missing'] / 
                 (station_prcp['total'] + station_prcp['missing']) > 0.5]
print(f"  Count: {len(mostly_missing)}")
print(mostly_missing.sort_values('missing', ascending=False).head(10))

Total stations: 94

Stations with ALL missing PRCP:  0
Stations with ALL zero PRCP:     2
Stations with any missing PRCP:  78

Stations where >50% PRCP is missing:
  Count: 3
             total  missing  valid  all_missing  all_zero  mean
STATION                                                        
72020200118   2313     2920   2313        False     False   0.0
72695024285   1693     2256   1693        False     False   0.0
99999994224    496      600    496        False     False   0.0


In [6]:
weather = pd.read_csv('oregon_weather_cleaned.csv', parse_dates=['DATE'])

problem_stations = ['72020200118', '72695024285', '99999994224']

for station in problem_stations:
    sdf = weather[weather['STATION'].astype(str) == station].sort_values('DATE')
    
    print(f"\n{'='*60}")
    print(f"Station: {station}")
    print(f"Name: {sdf['NAME'].iloc[0] if len(sdf) > 0 else 'Unknown'}")
    print(f"Total records:    {len(sdf):,}")
    print(f"Date range:       {sdf['DATE'].min().date()} to {sdf['DATE'].max().date()}")
    print(f"PRCP missing:     {sdf['PRCP'].isna().sum():,} ({sdf['PRCP'].isna().mean()*100:.1f}%)")
    print(f"PRCP zeros:       {(sdf['PRCP']==0).sum():,} ({(sdf['PRCP']==0).mean()*100:.1f}%)")
    print(f"PRCP valid:       {sdf['PRCP'].notna().sum():,}")
    print(f"\nPRCP by year (mean):")
    print(sdf.groupby(sdf['DATE'].dt.year)['PRCP'].agg(['mean','count','size']).head(20))


Station: 72020200118
Name: TILLAMOOK AIRPORT, OR US
Total records:    5,233
Date range:       2005-09-17 to 2021-12-31
PRCP missing:     2,920 (55.8%)
PRCP zeros:       2,313 (44.2%)
PRCP valid:       2,313

PRCP by year (mean):
      mean  count  size
DATE                   
2005   0.0      2     2
2006   0.0      6    27
2007   0.0    116   253
2008   0.0    157   366
2009   0.0    167   365
2010   0.0    133   365
2011   0.0    139   365
2012   0.0    150   362
2013   0.0    141   320
2014   0.0    149   336
2015   0.0    162   326
2016   0.0    146   359
2017   0.0    163   360
2018   0.0    173   365
2019   0.0    181   347
2020   0.0    167   360
2021   0.0    161   355

Station: 72695024285
Name: NEWPORT MUNICIPAL AIRPORT, OR US
Total records:    3,949
Date range:       2011-03-01 to 2021-12-31
PRCP missing:     2,256 (57.1%)
PRCP zeros:       1,693 (42.9%)
PRCP valid:       1,693

PRCP by year (mean):
      mean  count  size
DATE                   
2011   0.0    125   304
2012

In [7]:
weather = pd.read_csv('oregon_weather_cleaned.csv', parse_dates=['DATE'])

station_prcp = weather.groupby(['STATION', 'NAME'])['PRCP'].agg(
    valid=lambda x: x.notna().sum(),
    total='size',
    mean='mean',
    max='max',
    all_zero=lambda x: (x == 0).all(),
    pct_valid=lambda x: x.notna().mean() * 100
).reset_index()

bad_stations = station_prcp[
    station_prcp['all_zero'] |
    (station_prcp['pct_valid'] < 40)
]

print(f"Unreliable PRCP stations: {len(bad_stations)}")
print(bad_stations[['STATION','NAME','total','valid','pct_valid','mean','max','all_zero']].to_string())

Unreliable PRCP stations: 2
        STATION                                        NAME  total  valid  pct_valid  mean  max  all_zero
3   72036599999                    BROOKINGS AIRPORT, OR US      4      4      100.0   0.0  0.0      True
93  99999994236  KLAMATH FALLS INTERNATIONAL AIRPORT, OR US      1      1      100.0   0.0  0.0      True
